<a href="https://colab.research.google.com/github/frankettheofranckettheo/Advanced-ML-Project/blob/tp3/cnn_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Les Fondamentaux des CNN
## 1.2 Practical: Preparing the CIFAR-10 Data

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

# === 1. Préparation des données (Partie 1.2) ===

# 1. Charger le dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Nombre de classes
NUM_CLASSES = 10
INPUT_SHAPE = x_train.shape[1:] # (32, 32, 3)

# 2. Normaliser les valeurs des pixels vers [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 3. Convertir les labels en One-Hot Encoding
y_train = keras.utils.to_categorical(y_train, num_classes=NUM_CLASSES)
y_test = keras.utils.to_categorical(y_test, num_classes=NUM_CLASSES)

print(f"Input data shape: {INPUT_SHAPE}")

# TODO: Print the shape of the labels after conversion
print(f"Labels shape after conversion: {y_train.shape}")


# === 2. Implémentation CNN de base (Partie 2.1) ===

def build_basic_cnn(input_shape, num_classes):
    model = keras.Sequential([
        # Convolutional Layer 1
        keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),

        # Pooling Layer 1: Max Pooling 2x2
        # TODO: Add the Max Pooling layer
        keras.layers.MaxPooling2D(pool_size=(2, 2)),

        # Convolutional Layer 2
        # TODO: Add the second Conv2D layer
        keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),

        # Pooling Layer 2
        keras.layers.MaxPooling2D(pool_size=(2, 2)),

        # Flatten Layer
        keras.layers.Flatten(),

        # Dense Layer 1
        keras.layers.Dense(512, activation='relu'),

        # Output Layer
        keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_basic_cnn(INPUT_SHAPE, NUM_CLASSES)

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Entraînement du modèle
print("Début de l'entraînement...")
history = model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=10,
    validation_split=0.1
)

# TODO: Evaluate the model on x_test, y_test and display accuracy
print("\nEvaluation du modèle :")
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 46s 0us/step
Input data shape: (32, 32, 3)
Labels shape after conversion: (50000, 10)
Début de l'entraînement...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 92s 129ms/step - accuracy: 0.4120 - loss: 1.6231 - val_accuracy: 0.6098 - val_loss: 1.0984
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 142s 129ms/step - accuracy: 0.6295 - loss: 1.0411 - val_accuracy: 0.6556 - val_loss: 0.9967
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 141s 128ms/step - accuracy: 0.6975 - loss: 0.8605 - val_accuracy: 0.7148 - val_loss: 0.8408
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 141s 126ms/step - accuracy: 0.7485 - loss: 0.7189 - val_accuracy: 0.7058 - val_loss: 0.8576
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 89s 127ms/step - accuracy: 0.7934 - loss: 0.5962 - val_accuracy: 0.7214 - val_loss: 0.8344
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 143s 128ms/step - accuracy: 0.8323 - loss: 0.4816 - val_accuracy: 0.7264 - val_loss: 0.8584
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 141s 127ms/step - accuracy: 0.8712 - loss: 0.3755 - val_accuracy: 0.7290 - val_loss: 0.8974
Epoch 8/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 90s 128ms/step - accuracy: 0.9104 - lo

# 2.2 Exercise 2: Introduction to Residual Blocks (ResNets)

In [ ]:
def residual_block(x, filters, kernel_size=(3, 3), stride=1):
    # Main path
    y = keras.layers.Conv2D(filters, kernel_size, strides=stride, padding='same', activation='relu')(x)
    y = keras.layers.Conv2D(filters, kernel_size, padding='same')(y) # Pas d'activation ici avant l'addition

    # Skip Connection path
    if stride > 1:
        # Si on change la taille (stride > 1), on doit aussi redimensionner x
        x = keras.layers.Conv2D(filters, (1, 1), strides=stride)(x)

    # TODO: Complete the addition of the skip path with the main path
    z = keras.layers.Add()([x, y])

    z = keras.layers.Activation('relu')(z)
    return z

# TODO: Build a small architecture using 3 consecutive residual blocks
input_layer = keras.Input(shape=INPUT_SHAPE)
# Une première conv est souvent nécessaire pour projeter dans l'espace des features
x = keras.layers.Conv2D(32, (3,3), padding='same', activation='relu')(input_layer)

x = residual_block(x, 32)           # Bloc 1
x = residual_block(x, 64, stride=2) # Bloc 2 (réduit la taille de l'image)
x = residual_block(x, 64)           # Bloc 3

# Pour finir le modèle (exemple)
x = keras.layers.GlobalAveragePooling2D()(x)
output = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
resnet_model = keras.Model(inputs=input_layer, outputs=output)

# 3.2. Exercise 4: Neural Style Transfer

In [ ]:
# ... (chargement des images code précédent)

# 2. Load the pre-trained VGG16 model
# TODO: Complete code to load VGG16
vgg = keras.applications.VGG16(include_top=False, weights='imagenet')

vgg.trainable = False # Important: on n'entraîne pas VGG

# Définition des couches (fournies dans le PDF)
content_layers = ['block5_conv2']
style_layers = ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1', 'block5_conv1']

# La suite du code concerne l'extracteur et la boucle d'optimisation
# (comme montré dans le listing 4 du PDF)